In [ ]:
import torch
import functools

_original_torch_load = torch.load

@functools.wraps(_original_torch_load)
def _patched_torch_load(f, *args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _original_torch_load(f, *args, **kwargs)

torch.load = _patched_torch_load
print("✅ torch.load patched to use weights_only=False by default")


In [ ]:
# ===========================================================================
# CELL 1 – System setup: MuJoCo + display (needed for robosuite/LIBERO)
# ===========================================================================

import subprocess, sys, os

def run(cmd, check=False, **kwargs):
    """Run a shell command and print its output."""
    print(f"\n>>> {cmd}")
    result = subprocess.run(cmd, shell=True, check=check, **kwargs)
    if result.returncode != 0:
        print(f"[WARNING] exited {result.returncode}")
    return result

# ── System libs for offscreen MuJoCo rendering ───────────────────────────────
run("apt-get install -y --no-install-recommends "
    "xvfb x11-utils libosmesa6-dev libgl1-mesa-glx "
    "libglfw3 patchelf ffmpeg libglib2.0-0 2>&1 | tail -5")

# Set offscreen rendering env vars BEFORE any mujoco/robosuite import
os.environ["MUJOCO_GL"]       = "osmesa"
os.environ["PYOPENGL_PLATFORM"] = "osmesa"
os.environ["DISPLAY"]         = ":1"

# Start virtual framebuffer
run("Xvfb :1 -screen 0 1024x768x24 -ac +extension GLX &")




In [ ]:
# ===========================================================================
# CELL 2 – Install Python dependencies (order matters!)
# ===========================================================================

run("pip install -q --upgrade pip setuptools wheel")

# ── transformers from source (required for Qwen2.5-VL) ───────────────────────
run("pip install -q 'transformers>=4.52' accelerate")

# ── Qwen VL helper ───────────────────────────────────────────────────────────
run("pip install -q 'qwen-vl-utils>=0.0.10'")

# ── TRL (SFTTrainer) + extras ────────────────────────────────────────────────
run("pip install -q 'trl>=0.15' peft bitsandbytes")

# ── MuJoCo (free Python wheel) ───────────────────────────────────────────────
run("pip install -q 'mujoco>=3.0'")

# ── robosuite (pin to 1.4.1 — LIBERO uses robosuite.environments.manipulation
# which was reorganised in robosuite 2.x; the from-source clone installs latest
# and breaks LIBERO's imports) ──────────────────────────────────────────────────
run("pip install -q 'robosuite==1.4.1'")


# ── LIBERO ───────────────────────────────────────────────────────────────────
# IMPORTANT: Do NOT use `pip install libero` (wrong PyPI package) and do NOT
# use `pip install -e` (LIBERO's requirements.txt includes egl_probe/evdev
# which fail to build on Kaggle, aborting the install before registration).
#
# Correct approach: clone the repo, add its root to sys.path + PYTHONPATH.
# The importable `libero/` package lives inside the repo root — no pip needed.
LIBERO_DIR = "/kaggle/working/LIBERO"
run(f"git clone --depth=1 https://github.com/Lifelong-Robot-Learning/LIBERO.git "
    f"{LIBERO_DIR} 2>/dev/null || true")

!sed -i 's/torch.load(init_states_path)/torch.load(init_states_path, weights_only=False)/' \
    /kaggle/working/LIBERO/libero/libero/benchmark/__init__.py


# Real runtime deps only (skip requirements.txt entirely)
run("pip install -q bddl termcolor lxml Pillow h5py")

# Path registration IS the install
sys.path.insert(0, LIBERO_DIR)
os.environ["PYTHONPATH"] = LIBERO_DIR + ":" + os.environ.get("PYTHONPATH", "")

# LIBERO's libero/__init__.py has a bare input() call that fires on every import,
# asking where to store datasets.  In non-interactive subprocesses this raises
# EOFError.  LIBERO_DATA_PATH does NOT suppress it — the prompt comes first.
#
# Fix: patch the source file to skip the prompt entirely.
LIBERO_DATA = "/kaggle/working/LIBERO/libero/datasets"
os.makedirs(LIBERO_DATA, exist_ok=True)
os.environ["LIBERO_DATA_PATH"] = LIBERO_DATA

libero_init = os.path.join(LIBERO_DIR, "libero", "libero", "__init__.py")
with open(libero_init, "r") as f:
    init_src = f.read()

if 'input(' in init_src:
    import re
    # Replace answer=input(...)...path block, stop before set_libero_default_path
    patched = re.sub(
        r'answer\s*=\s*input\(.*?\).*?(?=\nset_libero_default_path|\nclass |\ndef |\Z)',
        ('# [patched by kaggle eval notebook]\n'
         '_libero_data_path = os.environ.get("LIBERO_DATA_PATH", "' + '/kaggle/working/LIBERO/libero/datasets' + '")'),
        init_src,
        flags=re.DOTALL,
    )
    patched = patched.replace('set_libero_default_path(path)',
                              'set_libero_default_path(_libero_data_path)')
    with open(libero_init, "w") as f:
        f.write(patched)
    print(f"Patched {libero_init}")
else:
    print("input() not found in __init__.py — already patched or structure changed")


# Verify
result = run(
    f"PYTHONPATH=\"{LIBERO_DIR}:$PYTHONPATH\" "
    f"LIBERO_DATA_PATH=\"{LIBERO_DATA}\" "
    "python -c \"from libero.libero import benchmark; print('libero OK')\""
)
if result.returncode != 0:
    # Print the relevant portion of __init__.py to diagnose
    with open(libero_init) as f:
        lines = f.readlines()
    print("__init__.py lines around any remaining input():")
    for i, line in enumerate(lines):
        if "input" in line or "answer" in line or "path" in line.lower():
            print(f"  {i:3d}: {line}", end="")
    raise RuntimeError(
        "LIBERO import still fails after patching.\n"
        f"  {LIBERO_DIR}: {os.listdir(LIBERO_DIR)}"
    )

# ── Write ~/.libero/config.yaml ──────────────────────────────────────────────
# LIBERO's get_libero_path() reads ~/.libero/config.yaml to find init_states.
# Our input() patch skipped the prompt that normally writes this file.
# The .pruned_init files are already bundled inside the cloned LIBERO repo at
# libero/libero/init_files/ — no separate download needed.
import pathlib
_libero_cfg_dir   = pathlib.Path.home() / ".libero"
_libero_cfg_file  = _libero_cfg_dir / "config.yaml"
_libero_init_files = os.path.join(LIBERO_DIR, "libero", "libero", "init_files")
_libero_cfg_dir.mkdir(parents=True, exist_ok=True)
with open(_libero_cfg_file, "w") as _f:
    _f.write(f"libero_base_path: {LIBERO_DATA}\n")
    _f.write(f"datasets: {os.path.join(LIBERO_DATA, 'datasets')}\n")
    _f.write(f"init_states: {_libero_init_files}\n")
print(f"Wrote {_libero_cfg_file}")
# Confirm the files are actually there
_sample = os.listdir(_libero_init_files) if os.path.isdir(_libero_init_files) else "MISSING"
print(f"init_files dir contents: {_sample[:5] if isinstance(_sample, list) else _sample}")



# ── LeRobot (pinned commit used by vla0-trl) ─────────────────────────────────
run("GIT_LFS_SKIP_SMUDGE=1 pip install -q "
    "\"git+https://github.com/huggingface/lerobot.git"
    "@f39652707caed42a7cd5ab36066da5663b9565eb\"")

# ── Misc utils ───────────────────────────────────────────────────────────────
run("pip install -q huggingface_hub einops timm imageio imageio-ffmpeg "
    "opencv-python-headless tomli")




In [ ]:
# ===========================================================================
# CELL 3 – Clone & install vla0-trl
# ===========================================================================

REPO_DIR = "/kaggle/working/vla0-trl"

run(f"git clone --depth=1 https://github.com/MilkClouds/vla0-trl.git {REPO_DIR} "
    f"2>/dev/null || (cd {REPO_DIR} && git pull)")

# Install core package (no-deps to avoid stomping on our carefully pinned stack)
run(f"pip install -q --no-deps -e '{REPO_DIR}'")
# Install [eval] extras — skip egl_probe/evdev which always fail on Kaggle.
# Parse pyproject.toml directly so we get exactly what [eval] needs.
try:
    import tomllib
except ImportError:
    import tomli as tomllib  # Python < 3.11 fallback

with open(os.path.join(REPO_DIR, "pyproject.toml"), "rb") as _f:
    _proj = tomllib.load(_f)
_eval_deps = _proj.get("project", {}).get("optional-dependencies", {}).get("eval", [])
_safe = [d for d in _eval_deps if not any(x in d.lower() for x in ("egl_probe", "egl-probe", "evdev"))]
if _safe:
    run("pip install -q " + " ".join(f"'{d}'" for d in _safe))
print(f"[eval] safe deps installed: {_safe}")
print(f"[eval] skipped: {[d for d in _eval_deps if d not in _safe]}")

# Make src/ importable
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
sys.path.insert(0, LIBERO_DIR)   # keep libero on path after all installs

print("✅  vla0-trl installed")


# ===========================================================================
# CELL 4 – Download Qwen2.5-VL-3B-Instruct checkpoint
# ===========================================================================
# vla0-trl fine-tunes Qwen2.5-VL-3B-Instruct on LIBERO data.
#
# ┌─────────────────────────────────────────────────────────────────────────┐
# │  CHECKPOINT OPTIONS (choose one):                                       │
# │                                                                         │
# │  A) YOUR OWN CHECKPOINT  ← set MODEL_PATH to the path of your          │
# │     checkpoint trained with vla0-trl's `scripts/train.py`.             │
# │     Example: "/kaggle/input/my-vla0-checkpoint/checkpoint-80000"       │
# │                                                                         │
# │  B) BASE MODEL ONLY  ← no fine-tuning; just sanity-check the pipeline  │
# │     Set MODEL_PATH = "Qwen/Qwen2.5-VL-3B-Instruct"                    │
# │     (will NOT produce good LIBERO scores, but the eval loop will run)  │
# └─────────────────────────────────────────────────────────────────────────┘

from huggingface_hub import snapshot_download

MODEL_ID   = "Qwen/Qwen2.5-VL-3B-Instruct"     # base model on HF Hub
MODEL_PATH = "/kaggle/working/qwen2.5-vl-3b"    # local download dir

# ⬇️  Change MODEL_PATH here if you have a fine-tuned checkpoint:
#   MODEL_PATH = "/kaggle/input/your-vla0-trl-checkpoint/checkpoint-80000"

if not os.path.isdir(MODEL_PATH):
    print(f"Downloading {MODEL_ID} → {MODEL_PATH} …")
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=MODEL_PATH,
        ignore_patterns=["*.msgpack", "flax_model*", "tf_model*"],
    )
    print("✅  Model downloaded")
else:
    print(f"✅  Model already at {MODEL_PATH}")


# ===========================================================================
# CELL 4b – Generate dataset_stats.json from lerobot/libero
# ===========================================================================
# eval.py requires dataset_stats.json (action min/max per dim) which is normally
# produced during training. We compute it by downloading ONLY the parquet files
# from lerobot/libero (no videos — parquet is ~20 MB total) and scanning actions.

import json
import numpy as np
from pathlib import Path
from huggingface_hub import snapshot_download

STATS_PATH = os.path.join(MODEL_PATH, "dataset_stats.json")

if os.path.exists(STATS_PATH):
    print(f"✅  dataset_stats.json already exists at {STATS_PATH}")
else:
    print("Computing dataset_stats.json from lerobot/libero parquet files …")
    print("(Downloading ~20 MB of parquet files, no videos)")

    # Download only parquet + metadata, skip videos
    lerobot_meta_dir = "/kaggle/working/lerobot_libero_meta"
    snapshot_download(
        repo_id="lerobot/libero",
        repo_type="dataset",
        local_dir=lerobot_meta_dir,
        ignore_patterns=["*.mp4", "videos/*"],   # skip video files (~500 MB)
    )

    # Read all parquet files and collect action statistics
    import glob
    parquet_files = sorted(glob.glob(os.path.join(lerobot_meta_dir, "data", "**", "*.parquet"), recursive=True))
    print(f"Found {len(parquet_files)} parquet files")

    # Use pandas for fast column-level reading
    run("pip install -q pyarrow pandas")
    import pandas as pd

    action_min = None
    action_max = None
    action_sum = None
    action_sum2 = None
    n_total = 0

    for pf in parquet_files:
        df = pd.read_parquet(pf, columns=["action"])
        # action column is a list of 7 floats per row
        actions = np.array(df["action"].tolist(), dtype=np.float32)  # (N, 7)
        batch_min = actions.min(axis=0)
        batch_max = actions.max(axis=0)
        if action_min is None:
            action_min  = batch_min
            action_max  = batch_max
            action_sum  = actions.sum(axis=0)
            action_sum2 = (actions ** 2).sum(axis=0)
        else:
            action_min  = np.minimum(action_min,  batch_min)
            action_max  = np.maximum(action_max,  batch_max)
            action_sum  += actions.sum(axis=0)
            action_sum2 += (actions ** 2).sum(axis=0)
        n_total += len(actions)

    action_mean = action_sum / n_total
    action_std  = np.sqrt(action_sum2 / n_total - action_mean ** 2)

    stats = {
        "action": {
            "min":  action_min.tolist(),
            "max":  action_max.tolist(),
            "mean": action_mean.tolist(),
            "std":  action_std.tolist(),
        }
    }

    with open(STATS_PATH, "w") as f:
        json.dump(stats, f, indent=2)

    print(f"✅  dataset_stats.json saved to {STATS_PATH}")
    print(f"   action dim=7 | n_frames={n_total}")
    print(f"   min:  {[f'{v:.3f}' for v in action_min]}")
    print(f"   max:  {[f'{v:.3f}' for v in action_max]}")




In [ ]:
# ===========================================================================
# CELL 5 – Configuration
# ===========================================================================

# ── Evaluation parameters (mirror vla0-trl README defaults) ──────────────────

TASK_SUITE       = "libero_spatial"   # libero_spatial | libero_object | libero_goal | libero_10
ACTION_HORIZON   = 8                  # execute N actions before re-querying model
ENSEMBLE_PRED    = 8                  # average N overlapping action chunks (set 1 to disable)
# NUM_EPISODES is not a vla0-trl eval.py argument.
# Episode count is fixed per task (50 in the paper); use --num_shards/--shard_id
# to parallelise, or just let it run all episodes for the chosen suite.
NUM_EPISODES     = 20
LOG_DIR          = "/kaggle/working/eval_results"
STATS_PATH       = os.path.join(MODEL_PATH, "dataset_stats.json")  # set in Cell 4b
TORCH_COMPILE    = False              # set True on capable GPU for ~20% speedup
SHARD_ID         = 0                  # shard index (for multi-GPU parallelism; single GPU → 0)
NUM_SHARDS       = 1                  # total shards (1 = no sharding)

os.makedirs(LOG_DIR, exist_ok=True)

# Prevent LIBERO's interactive dataset-path prompt in all child processes
LIBERO_DATA = "/kaggle/working/LIBERO/libero/datasets"
os.makedirs(LIBERO_DATA, exist_ok=True)
os.environ["LIBERO_DATA_PATH"] = LIBERO_DATA

print(f"""
Eval config:
  model_path      = {MODEL_PATH}
  task_suite      = {TASK_SUITE}
  action_horizon  = {ACTION_HORIZON}
  ensemble_pred   = {ENSEMBLE_PRED}
  num_episodes    = {NUM_EPISODES}
  log_dir         = {LOG_DIR}
  torch_compile   = {TORCH_COMPILE}
  shard           = {SHARD_ID}/{NUM_SHARDS}
""")




In [ ]:
# ===========================================================================
# CELL 6 – Smoke-test: make sure imports work
# ===========================================================================

import sys, os

# ── Ensure source dirs are on path (safe to call multiple times) ─────────────
for p in [REPO_DIR, os.path.join(REPO_DIR, "src"), LIBERO_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

print("Testing imports …")

import torch
print(f"  torch {torch.__version__}  | CUDA: {torch.cuda.is_available()}")

import transformers
print(f"  transformers {transformers.__version__}")

try:
    from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
    print("  ✅ Qwen2_5_VLForConditionalGeneration")
except ImportError:
    from transformers import Qwen2VLForConditionalGeneration as Qwen2_5_VLForConditionalGeneration
    from transformers import AutoProcessor
    print("  ✅ Qwen2VLForConditionalGeneration (fallback)")

try:
    from qwen_vl_utils import process_vision_info
    print("  ✅ qwen_vl_utils")
except ImportError:
    print("  ⚠  qwen_vl_utils not found")

# ── LIBERO – critical, debug thoroughly if missing ───────────────────────────
# Ensure path is set (safe to call again; Cell 2 may have been skipped on re-run)
if LIBERO_DIR not in sys.path:
    sys.path.insert(0, LIBERO_DIR)
os.environ["PYTHONPATH"] = LIBERO_DIR + ":" + os.path.join(REPO_DIR, "src") + ":" + os.environ.get("PYTHONPATH", "")

# Pre-set dataset path so LIBERO's __init__.py never calls input()
LIBERO_DATA = "/kaggle/working/LIBERO/libero/datasets"
os.makedirs(LIBERO_DATA, exist_ok=True)
os.environ["LIBERO_DATA_PATH"] = LIBERO_DATA

try:
    from libero.libero import benchmark, get_libero_path
    print(f"  ✅ libero  (benchmark, get_libero_path)")
except ModuleNotFoundError as e:
    print(f"  ✗  libero import failed: {e}")
    print(f"     sys.path = {[p for p in sys.path if 'kaggle' in p or 'libero' in p.lower()]}")
    print(f"     LIBERO_DIR contents: {os.listdir(LIBERO_DIR) if os.path.isdir(LIBERO_DIR) else 'DIR MISSING'}")
    raise RuntimeError(
        f"Could not import libero. Make sure Cell 2 ran successfully and "
        f"{LIBERO_DIR}/libero/ exists."
    )

try:
    import robosuite
    print(f"  ✅ robosuite {robosuite.__version__}")
except Exception as e:
    print(f"  ⚠  robosuite: {e}")

try:
    from rv_eval.evaluator import LiberoEvaluator
    print("  ✅ rv_eval.evaluator (vla0-trl)")
except Exception as e:
    print(f"  ⚠  rv_eval: {e}")

print("\n✅ Imports OK")

import pathlib
cfg = pathlib.Path.home() / ".libero/config.yaml"
LIBERO_BDDL = "/kaggle/working/LIBERO/libero/libero/bddl_files"
with open(cfg, "a") as f:
    f.write(f"bddl_files: {LIBERO_BDDL}\n")
print(open(cfg).read())








import json

STATS_PATH = "/kaggle/working/qwen2.5-vl-3b/dataset_stats.json"

with open(STATS_PATH) as f:
    stats = json.load(f)

print("Current keys:", list(stats.keys()))

# Rename "action" → "out_ori_act" if needed
if "action" in stats and "out_ori_act" not in stats:
    stats["out_ori_act"] = stats.pop("action")
    with open(STATS_PATH, "w") as f:
        json.dump(stats, f, indent=2)
    print("✅ Renamed 'action' → 'out_ori_act'")
else:
    print("Keys already correct or unexpected structure:", list(stats.keys()))




In [ ]:
# ===========================================================================
# CELL 7 – Run evaluation via vla0-trl's eval script
# ===========================================================================
# We call `scripts/eval.py` directly via subprocess so we stay close to the
# original repo's interface.  All parameters come from the config block above.
#
# vla0-trl eval.py signature (from the README):
#   python scripts/eval.py
#       --model_path   <checkpoint_dir>
#       --task_suite   <suite_name>
#       --action_horizon <int>
#       --ensemble_prediction <int>
#       --log_dir      <output_dir>
#       [--torch_compile]
#       [--skip_evaluated]
#       --shard_id <int> --num_shards <int>

compile_flag = "--torch_compile" if TORCH_COMPILE else ""
eval_script  = os.path.join(REPO_DIR, "scripts", "eval.py")

# Build PYTHONPATH so the subprocess finds libero + rv_eval regardless of
# whether pip install fully registered the packages.
extra_paths = ":".join([
    os.path.join(REPO_DIR, "src"),
    REPO_DIR,
    LIBERO_DIR,
    os.environ.get("PYTHONPATH", ""),
])

cmd = (
    f"PYTHONPATH='{extra_paths}' "
    f"MUJOCO_GL=osmesa PYOPENGL_PLATFORM=osmesa "
    f"LIBERO_DATA_PATH='{LIBERO_DATA}' "
    f"python {eval_script} "
    f"  --model_path {MODEL_PATH} "
    f"  --stats_path {STATS_PATH} "
    f"  --task_suite {TASK_SUITE} "
    f"  --action_horizon {ACTION_HORIZON} "
    f"  --ensemble_prediction {ENSEMBLE_PRED} "
    f"  --log_dir {LOG_DIR} "
    f"  {compile_flag} "
    f"  --shard_id {SHARD_ID} "
    f"  --num_shards {NUM_SHARDS} "
)

print("=" * 70)
print("Starting evaluation …")
print("=" * 70)
print(f"Command:\n  {cmd}\n")

# Run the eval; don't use check=True so we see partial results on error
proc = subprocess.run(cmd, shell=True, cwd=REPO_DIR)

if proc.returncode == 0:
    print("\n✅  Evaluation finished successfully")
else:
    print(f"\n⚠  Eval script exited with code {proc.returncode}")
    print("   See output above for errors.")




In [ ]:
# ===========================================================================
# CELL 8 – (Optional) Inline eval implementation
# ===========================================================================
# If the vla0-trl scripts/eval.py fails to run (e.g. API incompatibility),
# this cell implements the evaluation loop inline, faithfully following the
# vla0-trl approach: discretize actions as text tokens with Qwen2.5-VL.
#
# This is a stand-alone fallback – it does NOT need scripts/eval.py.
# Skip it if Cell 7 succeeded.

FALLBACK_EVAL = False  # ← set True to use this instead of scripts/eval.py

if FALLBACK_EVAL:
    import re
    import json
    import numpy as np
    from PIL import Image
    import torch
    from pathlib import Path

    # ── vla0 action tokenization constants (from vla0-trl src/rv_train/model.py) ──
    ACTION_DIM  = 7          # 7-DoF robot (ΔxΔyΔz + ΔrΔpΔyaw + gripper)
    NUM_BINS    = 256         # discretise each action dim into 256 bins
    ACTION_MIN  = -1.0        # continuous action range
    ACTION_MAX  =  1.0
    HORIZON     = ACTION_HORIZON  # how many future steps to predict at once

    def discretize_action(action: np.ndarray) -> list[int]:
        """Map continuous action ∈ [ACTION_MIN, ACTION_MAX] → bin index ∈ [0, NUM_BINS-1]."""
        clipped = np.clip(action, ACTION_MIN, ACTION_MAX)
        bins    = ((clipped - ACTION_MIN) / (ACTION_MAX - ACTION_MIN) * (NUM_BINS - 1)).round()
        return bins.astype(int).tolist()

    def undiscretize_action(bins: list[int]) -> np.ndarray:
        """Map bin index → continuous action."""
        arr = np.array(bins, dtype=np.float32)
        return arr / (NUM_BINS - 1) * (ACTION_MAX - ACTION_MIN) + ACTION_MIN

    def parse_action_response(text: str, action_dim: int = ACTION_DIM, horizon: int = HORIZON) -> np.ndarray | None:
        """
        Parse the model's text output into a (horizon, action_dim) numpy array.

        vla0-trl format: each timestep on a separate line as comma-separated integers
        e.g.  "128, 128, 130, 127, 129, 128, 200\n128, 128, 130, ..."
        """
        try:
            lines = [l.strip() for l in text.strip().split("\n") if l.strip()]
            actions = []
            for line in lines[:horizon]:
                nums = [int(x.strip()) for x in re.split(r"[,\s]+", line) if x.strip().lstrip("-").isdigit()]
                if len(nums) >= action_dim:
                    actions.append(undiscretize_action(nums[:action_dim]))
            if not actions:
                return None
            # Pad with last action if fewer lines than horizon
            while len(actions) < horizon:
                actions.append(actions[-1])
            return np.stack(actions[:horizon])  # (horizon, action_dim)
        except Exception:
            return None

    def build_prompt(instruction: str, image: Image.Image) -> list[dict]:
        """Build the Qwen2.5-VL chat message for VLA-0 inference."""
        system_msg = (
            "You are a robot manipulation controller. "
            f"Given an image observation and a task instruction, predict the next {HORIZON} "
            f"robot actions. Each action has {ACTION_DIM} dimensions (dx, dy, dz, droll, dpitch, dyaw, gripper). "
            f"Output exactly {HORIZON} lines, each line containing {ACTION_DIM} integers "
            f"in the range [0, {NUM_BINS-1}], separated by commas. Output nothing else."
        )
        user_content = [
            {"type": "image", "image": image},
            {"type": "text",  "text": f"Task: {instruction}\nPredict the next {HORIZON} actions:"},
        ]
        return [
            {"role": "system",    "content": system_msg},
            {"role": "user",      "content": user_content},
        ]

    # ── Load model ────────────────────────────────────────────────────────────
    print("Loading model for inline eval …")

    try:
        from transformers import Qwen2_5_VLForConditionalGeneration
        ModelClass = Qwen2_5_VLForConditionalGeneration
    except ImportError:
        from transformers import Qwen2VLForConditionalGeneration
        ModelClass = Qwen2VLForConditionalGeneration

    from transformers import AutoProcessor
    from qwen_vl_utils import process_vision_info

    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype  = torch.bfloat16 if device == "cuda" else torch.float32

    model = ModelClass.from_pretrained(
        MODEL_PATH,
        torch_dtype=dtype,
        device_map="auto",
    )
    model.eval()

    processor = AutoProcessor.from_pretrained(MODEL_PATH)
    print(f"✅  Model loaded on {device}")

    # ── Inference helper ──────────────────────────────────────────────────────
    @torch.inference_mode()
    def predict_actions(instruction: str, rgb_image: np.ndarray) -> np.ndarray | None:
        """
        Given a task instruction string and an (H, W, 3) uint8 RGB frame,
        return predicted actions of shape (action_horizon, action_dim) or None.
        """
        pil_img = Image.fromarray(rgb_image.astype(np.uint8))
        messages = build_prompt(instruction, pil_img)

        text_input   = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)

        inputs = processor(
            text   = [text_input],
            images = image_inputs,
            videos = video_inputs,
            return_tensors="pt",
            padding=True,
        ).to(device)

        # Generate (short – action tokens only)
        generated = model.generate(
            **inputs,
            max_new_tokens=HORIZON * (ACTION_DIM * 4 + 2),   # rough upper bound
            do_sample=False,
            temperature=None,
            top_p=None,
        )
        # Decode only the newly generated tokens
        prompt_len  = inputs["input_ids"].shape[1]
        new_tokens  = generated[0, prompt_len:]
        response    = processor.decode(new_tokens, skip_special_tokens=True)
        return parse_action_response(response)

    # ── LIBERO evaluation loop ────────────────────────────────────────────────
    from libero.libero import benchmark
    from libero.libero.envs import OffScreenRenderEnv

    results_all = {}
    MAX_STEPS   = 600   # max steps per episode (LIBERO standard)

    benchmark_dict = benchmark.get_benchmark_dict()
    task_suite     = benchmark_dict[TASK_SUITE]()
    num_tasks      = task_suite.n_tasks

    print(f"\nEvaluating {num_tasks} tasks from '{TASK_SUITE}' ({NUM_EPISODES} episodes each) …\n")

    for task_id in range(num_tasks):
        task      = task_suite.get_task(task_id)
        task_name = task.name
        task_bddl = task_suite.get_task_bddl_file(task_id)

        successes = []

        env_args = {
            "bddl_file_name": task_bddl,
            "camera_heights":  128,
            "camera_widths":   128,
            "camera_names":    "agentview",
        }

        for ep_idx in range(NUM_EPISODES):
            env     = OffScreenRenderEnv(**env_args)
            env.seed(ep_idx * 100)
            obs     = env.reset()
            success = False

            chunk_buffer = []   # action_horizon buffer (for temporal chunking)
            step = 0

            while step < MAX_STEPS:
                # Re-query the model when buffer is empty
                if not chunk_buffer:
                    rgb  = obs["agentview_image"]   # (128,128,3) uint8
                    acts = predict_actions(task_name, rgb)
                    if acts is None:
                        # Fallback: zero action (should rarely happen)
                        acts = np.zeros((ACTION_HORIZON, ACTION_DIM), dtype=np.float32)
                    chunk_buffer = list(acts)

                action  = chunk_buffer.pop(0)
                obs, reward, done, info = env.step(action)
                step += 1

                if done or info.get("success", False):
                    success = True
                    break

            env.close()
            successes.append(float(success))
            print(f"  [{TASK_SUITE}] task {task_id:02d}/{num_tasks-1}  ep {ep_idx+1:02d}  {'✓' if success else '✗'}")

        task_sr = np.mean(successes) * 100
        results_all[task_name] = {"success_rate": task_sr, "successes": successes}
        print(f"  → Task '{task_name}': {task_sr:.1f}%\n")

    # ── Summary ───────────────────────────────────────────────────────────────
    avg_sr = np.mean([v["success_rate"] for v in results_all.values()])
    print("=" * 60)
    print(f"  {TASK_SUITE} — Average Success Rate: {avg_sr:.1f}%")
    print("=" * 60)
    for name, r in results_all.items():
        print(f"  {name}: {r['success_rate']:.1f}%")

    # Save results as JSON
    results_path = os.path.join(LOG_DIR, f"{TASK_SUITE}_results.json")
    with open(results_path, "w") as f:
        json.dump({"task_suite": TASK_SUITE, "avg_success_rate": avg_sr, "tasks": results_all}, f, indent=2)
    print(f"\nResults saved → {results_path}")




In [ ]:
# ===========================================================================
# CELL 9 – Parse & display results (works for both Cell 7 and Cell 8)
# ===========================================================================

import glob, json, os
import numpy as np

print("\n" + "=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)

json_files = glob.glob(os.path.join(LOG_DIR, "**/*.json"), recursive=True) + \
             glob.glob(os.path.join(LOG_DIR, "*.json"))

if not json_files:
    print("No result JSON files found yet.")
else:
    for jf in sorted(json_files):
        try:
            with open(jf) as f:
                data = json.load(f)
            print(f"\n  File: {jf}")
            if "avg_success_rate" in data:
                print(f"  Suite: {data.get('task_suite', '?')}")
                print(f"  Average Success Rate: {data['avg_success_rate']:.1f}%")
                for task_name, metrics in data.get("tasks", {}).items():
                    sr = metrics.get("success_rate", metrics.get("sr", "?"))
                    print(f"    {task_name}: {sr}")
            else:
                print(json.dumps(data, indent=2)[:500])
        except Exception as e:
            print(f"  Could not read {jf}: {e}")

print("\nDone.")

In [ ]:
# !sed -i 's/torch.load(init_states_path)/torch.load(init_states_path, weights_only=False)/' \
#     /kaggle/working/LIBERO/libero/libero/benchmark/__init__.py